## Data loading

In [ ]:
# ── Set your file path here ───────────────────────────────
DATASET_PATH = "BanglishRev.json.gz"   # update to your local path

import os
file_size_mb = os.path.getsize(DATASET_PATH) / (1024 ** 2)
print(f"✅ File found: {DATASET_PATH}")
print(f"   Size: {file_size_mb:.1f} MB")

## Install Dependancies

In [ ]:
!pip install implicit scikit-learn pandas numpy scipy tabulate --quiet
print("✅ All packages ready.")

## Imports

In [ ]:
import json, math, warnings
import numpy  as np
import pandas as pd
import scipy.sparse as sp

from scipy.sparse.linalg          import svds
from sklearn.metrics.pairwise     import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate                     import tabulate
from google.colab                 import files
from sklearn.utils.extmath import randomized_svd as sklearn_rsvd

warnings.filterwarnings("ignore")
print("✅ Imports done.")

## Data Loading and Filtering

In [ ]:
import gzip
import orjson
import os
import time
import multiprocessing as mp
import numpy as np
import pandas as pd

# ── Config ────────────────────────────────────────────────
CHUNK_SIZE = 5_000
N_WORKERS  = max(1, mp.cpu_count() - 1)

print(f"📦  Dataset  : {DATASET_PATH}")
print(f"🧠  Workers  : {N_WORKERS}  |  Chunk size : {CHUNK_SIZE:,}")


# ── Step 1: Load compressed JSON ──────────────────────────
print("\n⏳ Loading JSON …", end=" ")
t0 = time.time()

with gzip.open(DATASET_PATH, "rb") as f:
    raw_data = orjson.loads(f.read())

print(f"done in {time.time()-t0:.1f}s  ({len(raw_data):,} products)")


# ── Step 2: Worker function ───────────────────────────────
def flatten_chunk(product_chunk: list) -> list:
    """
    Flattens a chunk of product dicts into flat row dicts.
    Only extracts the 5 columns needed by the active models.
    """
    rows = []
    for product in product_chunk:
        pid      = str(product.get("Product ID", ""))
        category = product.get("Category", "Unknown")
        reviews  = product.get("Reviews") or []

        if not pid:
            continue

        rows.extend([
            {
                "user_id":    str(r["Buyer ID"]),
                "product_id": pid,
                "rating":     r.get("Current Rating"),
                "category":   category,
                "timestamp":  r.get("Review Date"),
            }
            for r in reviews
            if r.get("Buyer ID") is not None
            and r.get("Current Rating") is not None
        ])

    return rows


# ── Step 3: Parallel flatten ──────────────────────────────
def chunked(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i: i + size]

chunks = list(chunked(raw_data, CHUNK_SIZE))
print(f"\n⏳ Flattening {len(chunks)} chunks across {N_WORKERS} workers …")

t1 = time.time()
with mp.Pool(processes=N_WORKERS) as pool:
    nested = pool.map(flatten_chunk, chunks)

all_rows = [row for chunk_result in nested for row in chunk_result]
print(f"✅ Flattening done in {time.time()-t1:.1f}s  ({len(all_rows):,} rows)")


# ── Step 4: Build DataFrame — only the 5 needed columns ──
print("\n⏳ Building DataFrame …", end=" ")
t2 = time.time()

df_raw = pd.DataFrame(all_rows, columns=[
    "user_id", "product_id", "rating", "category", "timestamp"
])

# Free raw data immediately
del all_rows, nested, raw_data
import gc; gc.collect()

print(f"done in {time.time()-t2:.1f}s")
print(f"\n✅ df_raw shape   : {df_raw.shape}")
print(f"   Unique users   : {df_raw['user_id'].nunique():,}")
print(f"   Unique items   : {df_raw['product_id'].nunique():,}")
print(f"   Columns kept   : {list(df_raw.columns)}")
df_raw.head(3)

## RAM sanity check

In [ ]:
# ================================================================================
# This cell audits memory and optionally drops heavy columns before preprocessing.
# ================================================================================

def df_memory_report(df: pd.DataFrame):
    """Prints per-column and total RAM usage."""
    mem = df.memory_usage(deep=True)
    total_mb = mem.sum() / 1024**2
    col_report = (
        mem[1:]                         # drop 'Index' entry
        .sort_values(ascending=False)
        .apply(lambda x: f"{x/1024**2:.1f} MB")
    )
    print("── DataFrame Memory Usage ───────────────")
    for col, usage in col_report.items():
        print(f"  {col:<20} {usage}")
    print(f"  {'TOTAL':<20} {total_mb:.1f} MB")
    print("─────────────────────────────────────────")

df_memory_report(df_raw)

# ── Optional: shrink string columns with categoricals ─────
# category, parent_cat, root_cat have very low cardinality
# → converting to Categorical saves ~10x memory on those columns
for col in ["category", "parent_cat", "root_cat"]:
    if col in df_raw.columns:
        df_raw[col] = df_raw[col].astype("category")

print("\nAfter categorical conversion:")
df_memory_report(df_raw)

## Pre-processing

In [ ]:
MIN_USER_INTERACTIONS = 10 # options tested: 5, 10, 15, 15, 20
MIN_ITEM_INTERACTIONS = 3  # options tested: 3,  3,  3,  5,  5

def preprocess(df: pd.DataFrame, min_user: int = 10, min_item: int = 3) -> pd.DataFrame:
    df = df.copy()

    # ── Cast rating to numeric ────────────────────────────
    if df["rating"].dtype == object:
        df["rating"] = df["rating"].astype(str).str.strip()
    df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

    # ── Parse timestamps ──────────────────────────────────
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="%Y-%m-%d", errors="coerce"
    )

    # ── Drop rows with missing rating OR timestamp ────────
    before = len(df)
    df.dropna(subset=["user_id", "product_id", "rating", "timestamp"], inplace=True)
    print(f"  Dropped {before - len(df):,} rows with missing rating/timestamp")

    # ── Clip to valid range and cast ──────────────────────
    df["rating"] = df["rating"].clip(1, 5).astype(np.float32)

    # ── Deduplicate: same user reviewing same product twice
    #    Keep highest rating (most positive signal) ────────
    before = len(df)
    df = (
        df.sort_values("rating", ascending=False)
          .drop_duplicates(subset=["user_id", "product_id"], keep="first")
    )
    print(f"  Dropped {before - len(df):,} duplicate (user, product) pairs")

    # ── Low-memory: category as Categorical dtype ─────────
    df["category"] = df["category"].astype("category")

    # ── Convergent iterative k-core filtering ─────────────
    # Runs until no more rows are removed (guaranteed every
    # user >= min_user and every item >= min_item).
    previous_size = -1
    iteration = 0
    while previous_size != len(df):
        previous_size = len(df)
        iteration += 1

        u_counts = df["user_id"].value_counts()
        df = df[df["user_id"].isin(u_counts[u_counts >= min_user].index)]

        i_counts = df["product_id"].value_counts()
        df = df[df["product_id"].isin(i_counts[i_counts >= min_item].index)]

        removed = previous_size - len(df)
        print(f"  Sparsity pass {iteration}: removed {removed:,} rows")

    df.reset_index(drop=True, inplace=True)

    # ── Sanity check ──────────────────────────────────────
    assert np.isfinite(df["rating"].values).all(), \
        "\u274c Non-finite ratings detected \u2014 check input data"
    assert not df.duplicated(subset=["user_id", "product_id"]).any(), \
        "\u274c Duplicate (user, product) pairs remain"

    return df


df = preprocess(df_raw, min_user=MIN_USER_INTERACTIONS, min_item=MIN_ITEM_INTERACTIONS)

print(f"\n\u2705 After preprocessing:")
print(f"   Rows         : {len(df):,}")
print(f"   Unique users : {df['user_id'].nunique():,}")
print(f"   Unique items : {df['product_id'].nunique():,}")
print(f"   Rating range : [{df['rating'].min():.0f}, {df['rating'].max():.0f}]")
print(f"   Memory usage : {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
df.head(3)


## 3-way temporal train / validation / test split

In [ ]:
def temporal_split_three_way(df: pd.DataFrame):
    """
    Returns train_df, val_df, test_df.
    All three are guaranteed to have no overlap by user-item pair.
    """
    df = df.copy()
    df.sort_values(["user_id", "timestamp"], inplace=True, na_position="last")

    # ── Rank interactions per user from the end ────────────
    # rank 1 = most recent, rank 2 = second most recent, etc.
    df["recency_rank"] = (
        df.groupby("user_id")
          .cumcount(ascending=False) + 1
    )

    test_mask  = df["recency_rank"] == 1
    val_mask   = df["recency_rank"] == 2
    train_mask = df["recency_rank"] >= 3

    train_df = df[train_mask].drop(columns="recency_rank").reset_index(drop=True)
    val_df   = df[val_mask  ].drop(columns="recency_rank").reset_index(drop=True)
    test_df  = df[test_mask ].drop(columns="recency_rank").reset_index(drop=True)

    return train_df, val_df, test_df


train_df, val_df, test_df = temporal_split_three_way(df)

# ── Coverage report ────────────────────────────────────────
total_users = df["user_id"].nunique()

# Users with >= 3 interactions appear in all three splits
# Users with exactly 2 appear in train + val only
# Users with exactly 1 appear in train only

train_only_users = set(train_df["user_id"]) - set(val_df["user_id"])
val_and_test     = set(val_df["user_id"]) & set(test_df["user_id"])

print(f"✅ Split complete")
print(f"{'─'*50}")
print(f"  {'Split':<10} {'Rows':>10}  {'Users':>10}")
print(f"{'─'*50}")
print(f"  {'Train':<10} {len(train_df):>10,}  {train_df['user_id'].nunique():>10,}")
print(f"  {'Val':<10} {len(val_df):>10,}  {val_df['user_id'].nunique():>10,}")
print(f"  {'Test':<10} {len(test_df):>10,}  {test_df['user_id'].nunique():>10,}")
print(f"{'─'*50}")
print(f"  Total unique users in df : {total_users:,}")
print(f"  Users in val AND test    : {len(val_and_test):,}  (full evaluation pipeline)")
print(f"  Users in train only      : {len(train_only_users):,}  (< 3 interactions)")

# ── Leakage check ─────────────────────────────────────────
# No (user, product) pair should appear in more than one split
train_pairs = set(zip(train_df["user_id"], train_df["product_id"]))
val_pairs   = set(zip(val_df["user_id"],   val_df["product_id"]))
test_pairs  = set(zip(test_df["user_id"],  test_df["product_id"]))

tv_overlap  = train_pairs & val_pairs
tt_overlap  = train_pairs & test_pairs
vt_overlap  = val_pairs   & test_pairs

print(f"\n── Leakage Check ─────────────────────────────────")
print(f"  Train ∩ Val  : {len(tv_overlap):,}  {'✅' if len(tv_overlap)  == 0 else '❌ LEAK'}")
print(f"  Train ∩ Test : {len(tt_overlap):,}  {'✅' if len(tt_overlap)  == 0 else '❌ LEAK'}")
print(f"  Val   ∩ Test : {len(vt_overlap):,}  {'✅' if len(vt_overlap)  == 0 else '❌ LEAK'}")

## Sparse Matrix for implicit matrix factorization

In [ ]:
import scipy.sparse as sp
import numpy as np

ALPHA = 40

# Derive dimensions from the existing index maps built in Cell 6
n_users = len(user2idx)
n_items = len(item2idx)

# Add integer index columns to train_df using Cell 6's maps
train_df['user_idx'] = train_df['user_id'].map(user2idx)
train_df['item_idx'] = train_df['product_id'].map(item2idx)

rows = train_df['user_idx'].values
cols = train_df['item_idx'].values
confidence_vals = (1 + ALPHA * train_df['rating'].values).astype(np.float32)

confidence_mat = sp.csr_matrix(
    (confidence_vals, (rows, cols)),
    shape=(n_users, n_items)
)
item_user_mat = confidence_mat.T.tocsr()

print(f"✅ Confidence matrix : {confidence_mat.shape[0]:,} users × {confidence_mat.shape[1]:,} items")
print(f"   Non-zeros         : {confidence_mat.nnz:,}")
print(f"   Alpha             : {ALPHA}")
print(f"   Confidence range  : [{confidence_vals.min():.0f}, {confidence_vals.max():.0f}]")

In [ ]:
required = {
    "matrix":   matrix,
    "user2idx": user2idx,
    "item2idx": item2idx,
    "idx2item": idx2item,
    "train_df": train_df,
    "test_df":  test_df,
}

for name, obj in required.items():
    print(f"  {name:<12} ✅  {type(obj).__name__}", end="")
    if hasattr(obj, 'shape'):
        print(f"  shape={obj.shape}", end="")
    elif hasattr(obj, '__len__'):
        print(f"  len={len(obj):,}", end="")
    print()

## Recommender Sytem Models

In [ ]:
!pip -q install implicit

import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.sparse as sp

# Use Colab's built-in sklearn
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder

# Implicit ALS
from implicit.als import AlternatingLeastSquares

print("✅ Environment ready")
print("NumPy:", np.__version__)

# ── Add integer index columns to train_df ─────────────────
train_df['user_idx'] = train_df['user_id'].map(user2idx)
train_df['item_idx'] = train_df['product_id'].map(item2idx)

print(f"✅ Index columns added: "
      f"{train_df['user_idx'].notna().sum():,} / {len(train_df):,} rows mapped")


# ══════════════════════════════════════════════════════════
# EXPLICIT MF — TruncatedSVD (sklearn, no surprise needed)
# ══════════════════════════════════════════════════════════
class ExplicitMFRecommender:

    def __init__(self, n_factors=64, random_state=42):
        self.n_factors    = n_factors
        self.random_state = random_state

    def fit(self, train_df, matrix=None, user2idx=None,
            item2idx=None, idx2item=None, **kwargs):

        self.user2idx  = user2idx
        self.item2idx  = item2idx
        self.idx2item  = idx2item
        self.user_seen = (
            train_df.groupby("user_id")["product_id"]
            .apply(set).to_dict()
        )

        self.model = TruncatedSVD(
            n_components = self.n_factors,
            random_state = self.random_state,
        )

        # user_factors: (n_users, k)
        # item_factors: (k, n_items)
        self.user_factors = self.model.fit_transform(matrix).astype(np.float32)
        self.item_factors = self.model.components_.astype(np.float32)  # (k, n_items)

        print(f"  user_factors : {self.user_factors.shape}")
        print(f"  item_factors : {self.item_factors.shape}")
        print(f"  Explained variance ratio sum: "
              f"{self.model.explained_variance_ratio_.sum():.4f}")
        return self

    def score_all_items(self, user_id: str) -> np.ndarray:
        u_idx = self.user2idx.get(user_id, -1)
        if u_idx < 0:
            return None
        # dot product: (k,) @ (k, n_items) → (n_items,)
        return self.user_factors[u_idx] @ self.item_factors


# ══════════════════════════════════════════════════════════
# IMPLICIT MF — Weighted ALS  (Hu, Koren, Volinsky 2008)
# ══════════════════════════════════════════════════════════
class ImplicitMFRecommender:

    def __init__(self, n_factors=64, iterations=20,
                 regularization=0.01, alpha=40, random_state=42):
        self.n_factors      = n_factors
        self.iterations     = iterations
        self.regularization = regularization
        self.alpha          = alpha
        self.random_state   = random_state

    def fit(self, train_df, matrix=None, user2idx=None,
        item2idx=None, idx2item=None, **kwargs):
      self.user2idx  = user2idx
      self.item2idx  = item2idx
      self.idx2item  = idx2item
      self.user_seen = (
          train_df.groupby("user_id")["product_id"]
          .apply(set).to_dict()
      )

      rows = train_df['user_idx'].values
      cols = train_df['item_idx'].values
      conf = (1 + self.alpha * train_df['rating'].values).astype(np.float32)

      # Pass user-item directly — implicit >= 0.6 expects this orientation
      user_item_conf = sp.csr_matrix(
          (conf, (rows, cols)),
          shape=(len(user2idx), len(item2idx)),
          dtype=np.float32,
      )

      self.model = AlternatingLeastSquares(
          factors        = self.n_factors,
          iterations     = self.iterations,
          regularization = self.regularization,
          random_state   = self.random_state,
          use_gpu        = False,
      )
      self.model.fit(user_item_conf, show_progress=True)

      # With user_item input: user_factors=(n_users, k), item_factors=(n_items, k)
      self.user_factors = np.array(self.model.user_factors, dtype=np.float32)
      self.item_factors = np.array(self.model.item_factors, dtype=np.float32)

      print(f"  user_factors: {self.user_factors.shape}")  # expect (40016, 64)
      print(f"  item_factors: {self.item_factors.shape}")  # expect (33141, 64)
      print(f"  alpha={self.alpha}")
      return self

    def score_all_items(self, user_id: str) -> np.ndarray:
        u_idx = self.user2idx.get(user_id, -1)
        if u_idx < 0:
            return None
        return self.user_factors[u_idx] @ self.item_factors.T


print("\n✅ ExplicitMFRecommender and ImplicitMFRecommender defined.")

## Train Recommendation Models

In [ ]:
import time

MODELS = {}

shared_fit_kwargs = dict(
    train_df = train_df,
    matrix   = matrix,
    user2idx = user2idx,
    item2idx = item2idx,
    idx2item = idx2item,
)

# ── Model 1: Explicit MF (TruncatedSVD) ──────────────────
print("⏳ Training ExplicitMF...")
t0 = time.time()
explicit_model = ExplicitMFRecommender(n_factors=64)
explicit_model.fit(**shared_fit_kwargs)
MODELS["ExplicitMF"] = explicit_model
print(f"✅ ExplicitMF done in {time.time()-t0:.1f}s")

# ── Model 2: Implicit MF (ALS) ────────────────────────────
print("\n⏳ Training ImplicitMF...")
t0 = time.time()
implicit_model = ImplicitMFRecommender(n_factors=64, iterations=20, alpha=40)
implicit_model.fit(**shared_fit_kwargs)
MODELS["ImplicitMF"] = implicit_model
print(f"✅ ImplicitMF done in {time.time()-t0:.1f}s")

print("\n🎯 All models trained. Ready for Cell 10 evaluation.")
print(f"   MODELS keys: {list(MODELS.keys())}")

## Multi-K Vectorized Evaluation

In [ ]:
import time
import numpy as np
import pandas as pd
from tabulate import tabulate
from tqdm import tqdm

K_VALUES = [5, 10, 15, 30, 50]

# ── Test set restricted to users seen during training ─────
test_known  = test_df[test_df["user_id"].isin(user2idx)].copy()
test_users  = test_known["user_id"].values
test_truths = test_known["product_id"].values
N_TEST      = len(test_users)

print(f"Test users : {N_TEST:,}  |  K values = {K_VALUES}\n")


# ──────────────────────────────────────────────────────────
# METRIC HELPER
# Computes Hit, MRR, NDCG at all K values in one pass.
# ──────────────────────────────────────────────────────────
def vectorized_metrics_multi_k(effective_ranks_0idx: np.ndarray,
                                k_values: list) -> dict:
    n          = len(effective_ranks_0idx)
    result     = {"N_eval": n}
    ranks_1idx = effective_ranks_0idx + 1

    for k in k_values:
        in_topk = effective_ranks_0idx < k
        result[f"Hit@{k}"]  = float(np.mean(in_topk.astype(np.float32)))
        result[f"MRR@{k}"]  = float(np.mean(
            np.where(in_topk, 1.0 / ranks_1idx, 0.0)
        ))
        result[f"NDCG@{k}"] = float(np.mean(
            np.where(in_topk, 1.0 / np.log2(ranks_1idx + 1), 0.0)
        ))

    return result


# ──────────────────────────────────────────────────────────
# EVALUATOR
# Works for any model that exposes score_all_items(user_id).
# Both ExplicitMFRecommender and ImplicitMFRecommender do.
# ──────────────────────────────────────────────────────────
def evaluate_model(model,
                   test_users:  np.ndarray,
                   test_truths: np.ndarray,
                   k_values:    list,
                   item2idx:    dict,
                   model_name:  str = "model") -> dict:

    n       = len(test_users)
    n_items = len(item2idx)

    effective_ranks = np.full(n, n_items, dtype=np.int32)
    truth_item_idx  = np.array(
        [item2idx.get(t, -1) for t in test_truths], dtype=np.int32
    )

    t0          = time.time()
    report_step = max(1, n // 20)

    for i, u in enumerate(test_users):
        t_idx = truth_item_idx[i]
        if t_idx < 0:
            continue

        scores = model.score_all_items(u)
        if scores is None:
            continue

        scores = scores.astype(np.float32, copy=True)

        # Mask training items — keep truth item unmasked
        for p in model.user_seen.get(u, set()):
            ci = item2idx.get(p, -1)
            if ci >= 0 and ci != t_idx:
                scores[ci] = -np.inf

        truth_score = scores[t_idx]
        if not np.isfinite(truth_score):
            continue

        effective_ranks[i] = int((scores > truth_score).sum())

        if (i + 1) % report_step == 0 or (i + 1) == n:
            elapsed = time.time() - t0
            rate    = (i + 1) / max(elapsed, 1e-6)
            eta     = (n - i - 1) / rate
            print(f"    {(i+1)/n*100:5.1f}%  |  {i+1:,}/{n:,}  |  "
                  f"{rate:.0f} users/s  |  ETA {eta:.0f}s", end="\r")

    print(f"    100.0%  |  {n:,}/{n:,}  |  "
          f"done in {time.time()-t0:.1f}s          ")

    return vectorized_metrics_multi_k(effective_ranks, k_values)


# ──────────────────────────────────────────────────────────
# RUN EVALUATION
# ──────────────────────────────────────────────────────────
results = {}

for name, model in MODELS.items():
    print(f"\n⏳  Evaluating  {name}")
    results[name] = evaluate_model(
        model       = model,
        test_users  = test_users,
        test_truths = test_truths,
        k_values    = K_VALUES,
        item2idx    = item2idx,
        model_name  = name,
    )
    m       = results[name]
    summary = "  ".join(
        f"Hit@{k}={m[f'Hit@{k}']:.4f}  NDCG@{k}={m[f'NDCG@{k}']:.4f}"
        for k in K_VALUES
    )
    print(f"    {summary}")


# ──────────────────────────────────────────────────────────
# RESULTS TABLES
# ──────────────────────────────────────────────────────────
print("\n" + "█" * 72)
print(f"{'█  FULL EVALUATION RESULTS  █':^72}")
print("█" * 72)

for k in K_VALUES:
    rows = sorted(
        [
            [name,
             m["N_eval"],
             round(m[f"Hit@{k}"],  4),
             round(m[f"MRR@{k}"],  4),
             round(m[f"NDCG@{k}"], 4)]
            for name, m in results.items()
        ],
        key=lambda x: x[4],
        reverse=True,
    )
    print(f"\n{'─' * 72}\n  K = {k}\n{'─' * 72}")
    print(tabulate(
        rows,
        headers=["Model", "N Eval", f"Hit@{k}", f"MRR@{k}", f"NDCG@{k}"],
        tablefmt="fancy_grid",
        floatfmt=".4f",
    ))
    print(f"  🥇  Best @ K={k} : {rows[0][0]}  "
          f"(NDCG={rows[0][4]:.4f}  Hit={rows[0][2]:.4f}  MRR={rows[0][3]:.4f})")

# NDCG summary across all K
print(f"\n{'─' * 72}\n  NDCG SUMMARY — all models × all K\n{'─' * 72}")
summary_rows = sorted(
    [[name] + [round(results[name][f"NDCG@{k}"], 4) for k in K_VALUES]
     for name in results],
    key=lambda r: sum(r[1:]) / len(K_VALUES),
    reverse=True,
)
print(tabulate(
    summary_rows,
    headers=["Model"] + [f"NDCG@{k}" for k in K_VALUES],
    tablefmt="fancy_grid",
    floatfmt=".4f",
))

best = summary_rows[0]
print(f"\n🏆  Overall Best : {best[0]}")
print("    NDCG values  : " + "  ".join(
    f"@{k}={v:.4f}" for k, v in zip(K_VALUES, best[1:])
))

# Save to CSV
pd.DataFrame([
    {"Model": name,
     **{f"{metric}@{k}": results[name][f"{metric}@{k}"]
        for metric in ["Hit", "MRR", "NDCG"]
        for k in K_VALUES}}
    for name in results
]).to_csv("mf_results_(20,5).csv", index=False)
print("\n✅ Results saved to mf_benchmark_results.csv")
